The objective of this program is to analyze a high-dimensional socio-economic and health dataset by applying Principal Component Analysis (PCA) to reduce dimensionality and extract the most significant underlying factors.
These principal components are then used as input features in a Logistic Regression model to predict the likelihood of child malnutrition.

The approach aims to:
Reduce redundancy and multicollinearity among variables
Improve computational efficiency
Identify latent structures influencing malnutrition
Build a predictive model for classification of malnutrition risk

This dataset is survey-style dataset designed to resemble large-scale Indian household surveys such as the National Family Health Survey.
It contains 100,000 observations, where each row represents a household or child with multiple socio-economic, health, and environmental attributes. The variables are structured around key development dimensions such as income, food security, health status, education, gender factors, sanitation, infrastructure, and climate risks. These features collectively capture the multidimensional causes of child malnutrition. The dataset includes both continuous variables (e.g., income, dietary diversity) and categorical/binary indicators (e.g., immunization status, access to clean water). A target variable, Malnutrition_Risk, is included as a binary outcome indicating whether a child is at high or low risk of malnutrition. (Although the data is synthetically generated, it follows realistic statistical distributions, making it suitable for applying techniques like PCA and logistic regression for analysis and modeling.)

In [20]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# Load data
df = pd.read_csv('sdg_malnutrition_dataset.csv')

# Split X and y
X = df.drop(columns=["Malnutrition_Risk"])
y = df["Malnutrition_Risk"]

# -------------------------------
# STEP 1: Split columns
# -------------------------------
threshold = 20
categorical_columns = []
continuous_columns = []

for col in X.columns:
    if X[col].nunique() < threshold:
        categorical_columns.append(col)
    else:
        continuous_columns.append(col)

# -------------------------------
# STEP 2: Encode categorical
# -------------------------------
df_encoded = pd.get_dummies(X[categorical_columns], drop_first=True)

# Combine with continuous (raw for now)
df_combined = pd.concat([X[continuous_columns], df_encoded], axis=1)

# -------------------------------
# STEP 3: Train-Test Split (FIRST)
# -------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    df_combined, y, test_size=0.2, random_state=42
)

# -------------------------------
# STEP 4: Scale (ONLY FIT ON TRAIN)
# -------------------------------
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# -------------------------------
# STEP 5: PCA (Sir’s method)
# -------------------------------
pca = PCA(n_components=10)

X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print("Train shape after PCA:", X_train_pca.shape)
print("Test shape after PCA:", X_test_pca.shape)

# -------------------------------
# STEP 6: Train Model
# -------------------------------
model = LogisticRegression(max_iter=1000)
model.fit(X_train_pca, y_train)

print("Accuracy:", model.score(X_test_pca, y_test))
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# -------------------------------
# STEP 7: Predictions
# -------------------------------
y_pred = model.predict(X_test_pca)

# -------------------------------
# STEP 8: Evaluation
# -------------------------------
print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Train shape after PCA: (80000, 10)
Test shape after PCA: (20000, 10)
Accuracy: 0.99845
Accuracy: 0.99845

Confusion Matrix:
[[19961     2]
 [   29     8]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     19963
           1       0.80      0.22      0.34        37

    accuracy                           1.00     20000
   macro avg       0.90      0.61      0.67     20000
weighted avg       1.00      1.00      1.00     20000

